# 02_top_tracks

DML: gold_top_tracks — Ranked tracks by play count.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("snapshot_date", "")
snapshot_date = dbutils.widgets.get("snapshot_date")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

fct   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.fct_plays")
dim_t = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_tracks").select("track_id", "track_name")
today = F.to_date(F.lit(snapshot_date))


def _ranked(label, cutoff):
    src = fct.filter(cutoff) if cutoff is not None else fct
    return (
        src.groupBy("track_id").agg(F.count("play_id").alias("play_count"))
        .join(dim_t, "track_id", "left")
        .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("play_count"))))
        .filter(F.col("rank") <= 50)
        .withColumn("period",        F.lit(label))
        .withColumn("snapshot_date", today)
        .select("snapshot_date", "period", "rank", "track_id", "track_name", "play_count")
    )


result = (
    _ranked("7d",  F.col("played_at") >= F.date_sub(today, 7))
    .unionByName(_ranked("30d", F.col("played_at") >= F.date_sub(today, 30)))
    .unionByName(_ranked("all", None))
)

upsert_delta(result, f"{CATALOG}.{GOLD_SCHEMA}.gold_top_tracks", ["snapshot_date", "period", "rank"])
display(result.orderBy("period", "rank"))